<img src="https://sfile.chatglm.cn/images-ppt/1eabcf97adfd.png" width="500" align="center">

# Свёрточные нейронные сети (CNN) — как машины «видят»

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + понимание нейросетей)
**Время прохождения:** ~90–120 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **почему** обычные полносвязные сети плохо работают с изображениями;
- увидите свёртку **визуально** — как фильтр «скользит» по картинке;
- **своими руками** создадите фильтры и увидите результат;
- построите CNN **с нуля** на PyTorch (без fastai!);
- обучите модель на датасете и визуализируете, **что она научилась видеть**;
- разберёте pooling, stride, padding — с картинками и интерактивом;
- сравните свою CNN с предобученной (transfer learning).

### Принцип этого блокнота

> Вы — автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких «чёрных ящиков».

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Почему не полносвязная сеть? | Проблема: слишком много параметров |
| 3 | Что такое свёртка? | Визуальное объяснение: фильтр скользит по картинке |
| 4 | Фильтры руками | Создаём фильтры сами: вертикальный, горизонтальный, размытие |
| 5 | Интерактивный свёрточный демонстратор | Слайдеры: размер фильтра, stride, padding |
| 6 | Строим CNN на PyTorch | Conv2d, ReLU, MaxPool2d — построчно |
| 7 | Dataset и DataLoader | Загрузка данных для обучения |
| 8 | Обучаем CNN | Цикл обучения с визуализацией loss |
| 9 | Оценка и визуализация | Confusion matrix, примеры предсказаний |
| 10 | Что видит CNN? | Визуализация фильтров и карт признаков |
| 11 | Pooling, Stride, Padding | Подробно с интерактивом |
| 12 | Transfer Learning | Предобученная модель vs наша |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Создание и обучение нейросети (низкоуровневый контроль) |
| **torchvision** | Датасеты (CIFAR-10) и трансформации изображений |
| **matplotlib** | Визуализация: картинки, графики, фильтры |
| **numpy** | Работа с массивами (для визуализации фильтров) |
| **ipywidgets** | Интерактивные виджеты |

> В этом блокноте мы используем **чистый PyTorch** — без fastai!
> Это важно: вы должны понимать, что происходит на каждом уровне.


In [ ]:
# ============================================================
# ШАГ 1: Установка и импорт
# ============================================================

import torch                    # основной фреймворк
import torch.nn as nn           # нейросетевые слои (Conv2d, Linear, ReLU...)
import torch.nn.functional as F # функции активации, pooling
import torch.optim as optim     # оптимизаторы (SGD, Adam)
import torchvision              # датасеты и трансформации
import torchvision.transforms as transforms  # конвейер предобработки

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import Counter
import time

# Настройка шрифтов для русского текста в графиках
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ CPU — обучение будет медленнее. Включите GPU в Colab!')

# Фиксируем random seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)
print('\n✅ Все библиотеки импортированы')

---
## Шаг 2. Почему не полносвязная сеть?

В блокноте 01 мы использовали полносвязную сеть (Linear) для классификации. Почему она плохо работает с изображениями?

### Проблема: слишком много параметров

Представьте картинку **224x224x3** (RGB):
- Пикселей: 224 × 224 × 3 = **150 528** входов
- Если первый скрытый слой 1000 нейронов → **150 528 000** весов!
- Это только первый слой!

### Две фундаментальные проблемы

1. **Не учитывается пространственная структура** — полносвязная сеть «разворачивает» картинку в вектор, теряя информацию о том, какие пиксели стоят рядом.

2. **Нет трансляционной инвариантности** — если кошка сдвинута на 5 пикселей вправо, полносвязная сеть видит это как СОВСЕМ ДРУГОЙ вход. А для нас — это та же кошка.

### Свёрточная сеть решает обе проблемы!


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Полносвязная vs Свёрточная сеть
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ---- Полносвязная сеть ----
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Полносвязная сеть\n(каждый с каждым)', fontsize=14, fontweight='bold', color='red')

# Входные пиксели
for i in range(8):
    y = 7.5 - i * 0.7
    ax.add_patch(plt.Rectangle((0.5, y-0.25), 1.5, 0.5, fc='lightblue', ec='steelblue', lw=1.5))
    ax.text(1.25, y, f'px{i}', ha='center', va='center', fontsize=8)

# Скрытый слой
for i in range(6):
    y = 7.0 - i * 0.9
    ax.add_patch(plt.Rectangle((4.5, y-0.25), 1.5, 0.5, fc='lightyellow', ec='orange', lw=1.5))
    ax.text(5.25, y, f'h{i}', ha='center', va='center', fontsize=8)

# Все связи (каждый с каждым!)
for i in range(8):
    for j in range(6):
        y1 = 7.5 - i * 0.7
        y2 = 7.0 - j * 0.9
        ax.plot([2.0, 4.5], [y1, y2], 'gray', alpha=0.2, lw=0.5)

ax.text(5, 1.5, '150+ миллионов весов!\n(для 224x224x3)', 
        ha='center', fontsize=11, color='red', fontweight='bold',
        bbox=dict(boxstyle='round', fc='mistyrose'))

# ---- Свёрточная сеть ----
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Свёрточная сеть\n(локальные связи)', fontsize=14, fontweight='bold', color='green')

# Входные пиксели
for i in range(8):
    y = 7.5 - i * 0.7
    ax.add_patch(plt.Rectangle((0.5, y-0.25), 1.5, 0.5, fc='lightblue', ec='steelblue', lw=1.5))
    ax.text(1.25, y, f'px{i}', ha='center', va='center', fontsize=8)

# Фильтры (локальные связи!)
filter_colors = ['#FF6B6B', '#4ECDC4', '#FFD93D']
for f_idx in range(3):
    x_base = 4.0 + f_idx * 1.8
    for i in range(4):  # фильтр видит только 4 пикселя
        y = 6.8 - i * 1.2
        ax.add_patch(plt.Rectangle((x_base, y-0.25), 1.3, 0.5, 
                                   fc=filter_colors[f_idx], ec='gray', lw=1.5, alpha=0.7))
        # Связи только с «своими» пикселями
        for j in range(3):
            y_px = 7.5 - (i+j) * 0.7
            if 0 < y_px < 9:
                ax.plot([2.0, x_base], [y_px, y], color=filter_colors[f_idx], alpha=0.5, lw=1.5)

ax.text(5, 1.5, 'Сотни/тысячи весов\n(разделяемые параметры!)', 
        ha='center', fontsize=11, color='green', fontweight='bold',
        bbox=dict(boxstyle='round', fc='honeydew'))

plt.suptitle('Почему CNN лучше для изображений?', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print('Ключевое отличие: в CNN фильтр — один и тот же — применяется')
print('ко ВСЕМ позициям картинки. Это называется "разделяемые веса".')
print('Вместо 150 млн весов — всего несколько тысяч!')

---
## Шаг 3. Что такое свёртка?

Свёртка (convolution) — это главная операция в CNN. Давайте разберём её пошагово.

### Аналогия: фонарик в тёмной комнате

Представьте, что вы в тёмной комнате с маленьким фонариком:
- Фонарик освещает только **маленький участок** стены (например, 3x3 пикселя)
- Вы водите фонариком по стене (скользите фильтром по картинке)
- В каждом положении вы «замечаете» определённый паттерн (линия, угол, текстура)
- Результат — **карта признаков** (feature map), где ярче = сильнее паттерн

### Математика свёртки (простыми словами)

```
Для каждой позиции фильтра на картинке:
  1. Умножить пиксели картинки на веса фильтра (поэлементно)
  2. Сложить все произведения → одно число
  3. Записать это число в карту признаков
  4. Сдвинуть фильтр на 1 пиксель → повторить
```


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Как свёртка работает (пошагово)
# ============================================================
# Показываем, как фильтр 3x3 скользит по картинке 5x5

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Простая «картинка» 5x5
image = np.array([
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
], dtype=float)

# Вертикальный фильтр (детектор вертикальных линий)
kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

# Показываем 4 шага свёртки
positions = [(0, 0), (0, 2), (2, 0), (2, 2)]

for idx, (r, c) in enumerate(positions):
    ax = axes[idx]
    
    # Рисуем картинку
    ax.imshow(image, cmap='gray', vmin=-3, vmax=3)
    
    # Выделяем область фильтра (красная рамка)
    rect = plt.Rectangle((c-0.5, r-0.5), 3, 3, 
                         fill=False, ec='red', lw=3)
    ax.add_patch(rect)
    
    # Показываем значения фильтра
    for i in range(3):
        for j in range(3):
            ax.text(c+j, r+i, f'{kernel[i,j]:.0f}', 
                   ha='center', va='center', fontsize=8, color='red', fontweight='bold')
    
    # Вычисляем результат свёртки в этой позиции
    region = image[r:r+3, c:c+3]
    result = np.sum(region * kernel)
    
    ax.set_title(f'Позиция ({r},{c})\nСумма = {result:.1f}', fontsize=11)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))

plt.suptitle('Свёртка: фильтр 3x3 скользит по картинке 5x5', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Полная свёртка через цикл (показываем как это работает)
print('Полная карта признаков (результат свёртки):')
output_size = image.shape[0] - kernel.shape[0] + 1
feature_map = np.zeros((output_size, output_size))
for i in range(output_size):
    for j in range(output_size):
        region = image[i:i+3, j:j+3]
        feature_map[i, j] = np.sum(region * kernel)
        
print(feature_map)
print()
print('Белые области = вертикальные линии обнаружены!')
print('Чёрные области = вертикальных линий нет.')

---
## Шаг 4. Создаём фильтры своими руками

Фильтр — это маленькая матрица чисел. Разные фильтры «видят» разные паттерны:
- **Вертикальный** — обнаруживает вертикальные линии
- **Горизонтальный** — обнаруживает горизонтальные линии
- **Размытие** — усредняет соседние пиксели
- **Повышение резкости** — усиливает контраст
- **Границы (edge)** — обнаруживает перепады яркости

В CNN эти фильтры **не задаёт человек** — сеть **сама учится** нужным фильтрам!
Но сначала давайте попробуем ручные фильтры, чтобы понять принцип.


In [ ]:
# ============================================================
# РУЧНЫЕ ФИЛЬТРЫ: применяем к реальной картинке
# ============================================================
# Загружаем простую картинку из scipy (встроенная)
from PIL import Image
import requests

# Создаём тестовое изображение: шахматная доска + круг
img_size = 64
test_img = np.zeros((img_size, img_size), dtype=np.float32)

# Шахматная доска в левой половине
for i in range(img_size):
    for j in range(img_size // 2):
        if (i // 8 + j // 8) % 2 == 0:
            test_img[i, j] = 1.0

# Круг в правой половине
cy, cx = img_size // 2, 3 * img_size // 4
for i in range(img_size):
    for j in range(img_size):
        if (i - cy)**2 + (j - cx)**2 < (img_size // 5)**2:
            test_img[i, j] = 0.8

# Определяем фильтры
filters = {
    'Оригинал': np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]]),  # тождественный
    'Вертикальные\nлинии': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float),
    'Горизонтальные\nлинии': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float),
    'Размытие': np.ones((3, 3), dtype=float) / 9,
    'Резкость': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
    'Границы\n(edge)': np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=float),
}

# Применяем каждый фильтр
def apply_filter(image, kernel):
    """Применяет свёртку вручную (без PyTorch, чтобы видеть каждый шаг).
    
    Args:
        image: 2D массив (H x W)
        kernel: 2D массив (kH x kW)
    
    Returns:
        2D массив — карта признаков
    """
    kh, kw = kernel.shape
    h, w = image.shape
    output_h = h - kh + 1
    output_w = w - kw + 1
    output = np.zeros((output_h, output_w))
    
    for i in range(output_h):
        for j in range(output_w):
            # Вырезаем область размером с фильтр
            region = image[i:i+kh, j:j+kw]
            # Поэлементное умножение + сумма
            output[i, j] = np.sum(region * kernel)
    
    return output

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (name, kernel) in enumerate(filters.items()):
    ax = axes[idx // 3][idx % 3]
    
    if name == 'Оригинал':
        ax.imshow(test_img, cmap='gray', vmin=0, vmax=1)
    else:
        result = apply_filter(test_img, kernel)
        ax.imshow(result, cmap='gray')
    
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Показываем матрицу фильтра
    if name != 'Оригинал':
        filter_text = '\n'.join([' '.join([f'{v:5.1f}' for v in row]) for row in kernel])
        ax.text(0.5, -0.05, f'Фильтр:\n{filter_text}', 
               transform=ax.transAxes, fontsize=7, ha='center', va='top',
               family='monospace')

plt.suptitle('Ручные фильтры → результат свёртки', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('В CNN фильтры НЕ задаются вручную — сеть УЧИТСЯ им!')
print('Но понимание ручных фильтров помогает представить, что делает сеть.')

---
## Шаг 5. Интерактивный свёрточный демонстратор

Поиграйте с параметрами свёртки и увидите, как меняется результат:
- **Размер фильтра** — 3x3, 5x5, 7x7 (больше = видит больший контекст)
- **Stride** — шаг сдвига фильтра (1 = каждый пиксель, 2 = через один)
- **Padding** — дополнение нулями по краям (сохраняет размер)


In [ ]:
# ============================================================
# ИНТЕРАКТИВ: Свёрточный демонстратор
# ============================================================
# Выбираем реальную картинку из CIFAR-10 (скачаем ниже)
# Пока используем синтетическую

# Создаём интересную тестовую картинку
test_img_interactive = np.zeros((16, 16), dtype=np.float32)
# Диагональная линия
for i in range(16):
    test_img_interactive[i, i] = 1.0
# Горизонтальная линия
test_img_interactive[8, :] = 0.7

# Виджеты
kernel_size_w = widgets.IntSlider(value=3, min=3, max=7, step=2, 
                                   description='Фильтр:', layout=widgets.Layout(width='300px'))
stride_w = widgets.IntSlider(value=1, min=1, max=3, step=1,
                              description='Stride:', layout=widgets.Layout(width='300px'))
padding_w = widgets.IntSlider(value=0, min=0, max=3, step=1,
                               description='Padding:', layout=widgets.Layout(width='300px'))
filter_type_w = widgets.Dropdown(
    options=['Вертикальный', 'Горизонтальный', 'Диагональный', 'Размытие', 'Границы'],
    value='Вертикальный',
    description='Тип фильтра:',
    layout=widgets.Layout(width='300px')
)

output = widgets.Output()

def demo_convolution(kernel_size, stride, padding, filter_type):
    """Интерактивная демонстрация свёртки с настраиваемыми параметрами."""
    with output:
        clear_output(wait=True)
        
        # Создаём фильтр нужного размера
        if filter_type == 'Вертикальный':
            k = np.zeros((kernel_size, kernel_size))
            k[:, -1] = 1
            k[:, 0] = -1
        elif filter_type == 'Горизонтальный':
            k = np.zeros((kernel_size, kernel_size))
            k[-1, :] = 1
            k[0, :] = -1
        elif filter_type == 'Диагональный':
            k = np.zeros((kernel_size, kernel_size))
            np.fill_diagonal(k, 1)
            k = k - k.mean()
        elif filter_type == 'Размытие':
            k = np.ones((kernel_size, kernel_size)) / (kernel_size ** 2)
        elif filter_type == 'Границы':
            k = -np.ones((kernel_size, kernel_size))
            k[kernel_size//2, kernel_size//2] = kernel_size**2 - 1
        
        # Добавляем padding
        if padding > 0:
            padded = np.pad(test_img_interactive, padding, mode='constant', constant_values=0)
        else:
            padded = test_img_interactive
        
        # Вычисляем размер выхода
        h_in, w_in = padded.shape
        h_out = (h_in - kernel_size) // stride + 1
        w_out = (w_in - kernel_size) // stride + 1
        
        # Применяем свёртку
        result = np.zeros((h_out, w_out))
        for i in range(h_out):
            for j in range(w_out):
                r = i * stride
                c = j * stride
                region = padded[r:r+kernel_size, c:c+kernel_size]
                result[i, j] = np.sum(region * k)
        
        # Визуализация
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Оригинал
        axes[0].imshow(test_img_interactive, cmap='gray', vmin=0, vmax=1)
        axes[0].set_title(f'Вход: {test_img_interactive.shape[0]}x{test_img_interactive.shape[1]}', fontsize=11)
        axes[0].axis('off')
        
        # Фильтр
        axes[1].imshow(k, cmap='RdBu_r', vmin=-1, vmax=1)
        axes[1].set_title(f'Фильтр {kernel_size}x{kernel_size}\n({filter_type})', fontsize=11)
        axes[1].axis('off')
        for i in range(kernel_size):
            for j in range(kernel_size):
                axes[1].text(j, i, f'{k[i,j]:.1f}', ha='center', va='center', fontsize=7)
        
        # Результат
        axes[2].imshow(result, cmap='gray')
        axes[2].set_title(f'Выход: {h_out}x{w_out}\n(stride={stride}, pad={padding})', fontsize=11)
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Формула размера выхода
        print(f'Формула: output = (input - kernel + 2*padding) / stride + 1')
        print(f'Вычисление: ({h_in} - {kernel_size} + 2*{padding}) / {stride} + 1 = {h_out}')
        print(f'Размер входа:  {test_img_interactive.shape[0]}x{test_img_interactive.shape[1]}')
        print(f'С padding:     {h_in}x{w_in}')
        print(f'Размер выхода: {h_out}x{w_out}')

widgets.interact(demo_convolution, 
                 kernel_size=kernel_size_w, 
                 stride=stride_w, 
                 padding=padding_w,
                 filter_type=filter_type_w)
display(output)

### Поиграйте с параметрами!

- **Stride=1, Padding=0** → выход МЕНЬШЕ входа (стандартный вариант)
- **Stride=2, Padding=0** → выход в 2 раза МЕНЬШЕ (downsampling)
- **Stride=1, Padding=1** → выход РАВЕН входу (при kernel=3) — «same padding»
- **Stride=1, Padding=2** → выход БОЛЬШЕ входа (редко используется)

> **Формула размера выхода:** `output = (input - kernel + 2*padding) / stride + 1`

---

---
## Шаг 6. Строим CNN на PyTorch

Теперь мы построим настоящую CNN **с нуля** — без fastai, только PyTorch!

### Архитектура нашей сети

```
Вход (3x32x32 RGB картинка)
  → Conv2d(3, 16, 3)     → 16 карт признаков 30x30
  → ReLU                  → активация
  → MaxPool2d(2)          → 16 карт признаков 15x15
  → Conv2d(16, 32, 3)    → 32 карты признаков 13x13
  → ReLU                  → активация
  → MaxPool2d(2)          → 32 карты признаков 6x6
  → Flatten               → вектор 32*6*6 = 1152
  → Linear(1152, 64)      → 64 нейрона
  → ReLU                  → активация
  → Linear(64, 10)        → 10 классов (CIFAR-10)
```

Каждый слой делает что-то конкретное. Давайте разберём каждый!


In [ ]:
# ============================================================
# ШАГ 6: Определяем CNN
# ============================================================
# nn.Module — базовый класс для всех нейросетей в PyTorch
# Мы наследуемся от него и определяем:
#   __init__() — какие слои будут в сети
#   forward()  — в каком порядке данные проходят через слои

class SimpleCNN(nn.Module):
    """Простая свёрточная нейронная сеть для классификации CIFAR-10.
    
    Архитектура:
      Conv(3→16) → ReLU → Pool → Conv(16→32) → ReLU → Pool → FC → FC
    
    Итого ~75 000 параметров (вместо 150+ миллионов у полносвязной!)
    """
    
    def __init__(self):
        # Обязательно вызываем __init__ родителя!
        super(SimpleCNN, self).__init__()
        
        # ---- Свёрточные слои ----
        # Conv2d(in_channels, out_channels, kernel_size)
        # in_channels=3  — RGB (3 канала)
        # out_channels=16 — на выходе 16 карт признаков
        # kernel_size=3   — фильтр 3x3
        # padding=1       — добавляем 1 пиксель нулей по краям (same padding)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, 
                               kernel_size=3, padding=1)
        
        # Второй свёрточный слой:
        # in_channels=16  — с выхода conv1
        # out_channels=32 — удваиваем количество карт признаков
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, 
                               kernel_size=3, padding=1)
        
        # ---- Pooling слой ----
        # MaxPool2d(2) — берёт максимум в окне 2x2
        # Уменьшает размер картинки ВДВОЕ
        # 32x32 → 16x16 → 8x8
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # ---- Полносвязные слои (классификатор) ----
        # После conv1+pool: 16 карт 16x16 = 16*16*16 = 4096
        # После conv2+pool: 32 карты 8x8 = 32*8*8 = 2048
        # Но! С padding=1 размер не меняется после свёртки,
        # меняется только после pool: 32x32→16x16→8x8
        self.fc1 = nn.Linear(32 * 8 * 8, 64)   # 2048 → 64
        self.fc2 = nn.Linear(64, 10)             # 64 → 10 классов
        
        # ---- Dropout (регуляризация) ----
        # Случайно «выключает» 25% нейронов при обучении
        # Предотвращает переобучение (overfitting)
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        """Прямой проход: данные идут от входа к выходу.
        
        Args:
            x: тензор формы (batch_size, 3, 32, 32) — батч картинок
        
        Returns:
            тензор формы (batch_size, 10) — логиты для 10 классов
        """
        # ---- Блок 1: Conv + ReLU + Pool ----
        # x: (B, 3, 32, 32) → conv1 → (B, 16, 32, 32) → pool → (B, 16, 16, 16)
        x = self.pool(F.relu(self.conv1(x)))
        
        # ---- Блок 2: Conv + ReLU + Pool ----
        # x: (B, 16, 16, 16) → conv2 → (B, 32, 16, 16) → pool → (B, 32, 8, 8)
        x = self.pool(F.relu(self.conv2(x)))
        
        # ---- Разворачиваем в вектор ----
        # x: (B, 32, 8, 8) → (B, 2048)
        x = x.view(x.size(0), -1)  # -1 = «автоматически посчитай размер»
        
        # ---- Классификатор ----
        x = self.dropout(x)              # регуляризация
        x = F.relu(self.fc1(x))          # 2048 → 64
        x = self.dropout(x)              # регуляризация
        x = self.fc2(x)                  # 64 → 10 (логиты)
        
        return x

# Создаём модель и перемещаем на GPU/CPU
model = SimpleCNN().to(device)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Модель: SimpleCNN')
print(f'Всего параметров: {total_params:,}')
print(f'Обучаемых: {trainable_params:,}')
print(f'Устройство: {device}')
print()

# Визуализируем архитектуру
print('Архитектура:')
print(model)

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Схема архитектуры CNN
# ============================================================

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')

# Блоки архитектуры
blocks = [
    (0.2,  'Вход\n3x32x32',       'lightblue',   1.5),
    (1.8,  'Conv1\n3→16',         '#FFB3BA',     1.3),
    (3.2,  'ReLU',                  '#FFFFBA',     0.7),
    (4.2,  'Pool\n→16x16',        '#BAFFC9',     1.1),
    (5.6,  'Conv2\n16→32',        '#FFB3BA',     1.3),
    (7.0,  'ReLU',                  '#FFFFBA',     0.7),
    (8.0,  'Pool\n→8x8',          '#BAFFC9',     1.1),
    (9.4,  'Flatten\n→2048',      '#BAE1FF',     1.3),
    (11.0, 'FC1\n→64',            '#E8BAFF',     1.1),
    (12.4, 'ReLU',                  '#FFFFBA',     0.7),
    (13.4, 'FC2\n→10',            '#E8BAFF',     1.1),
    (14.8, 'Выход\n10 классов',   'lightblue',   1.0),
]

for x, text, color, w in blocks:
    ax.add_patch(plt.Rectangle((x, 2.0), w, 2.0, fc=color, ec='steelblue', lw=1.5, alpha=0.8))
    ax.text(x + w/2, 3.0, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Стрелки
for i in range(len(blocks)-1):
    x1 = blocks[i][0] + blocks[i][3]  # конец текущего блока
    x2 = blocks[i+1][0]               # начало следующего
    ax.annotate('', xy=(x2, 3.0), xytext=(x1, 3.0),
               arrowprops=dict(arrowstyle='->', lw=1.5, color='steelblue'))

# Подписи размерностей
sizes = ['3x32x32', '16x32x32', '16x32x32', '16x16x16', '32x16x16', 
         '32x16x16', '32x8x8', '2048', '64', '64', '10', '10']
for i, (x, text, color, w) in enumerate(blocks):
    ax.text(x + w/2, 1.5, sizes[i], ha='center', fontsize=7, color='gray', style='italic')

ax.set_title('Архитектура SimpleCNN', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Шаг 7. Dataset и DataLoader

Мы используем **CIFAR-10** — стандартный датасет из 60 000 картинок 32x32 (50 000 train + 10 000 test), 10 классов.

| Класс | Пример |
|-------|--------|
| airplane | ✈️ |
| automobile | 🚗 |
| bird | 🐦 |
| cat | 🐱 |
| deer | 🦌 |
| dog | 🐶 |
| frog | 🐸 |
| horse | 🐴 |
| ship | 🚢 |
| truck | 🚛 |

Маленький размер (32x32) = быстрое обучение — идеально для обучения!


In [ ]:
# ============================================================
# ШАГ 7: Загрузка CIFAR-10
# ============================================================
# torchvision.datasets.CIFAR10 автоматически скачает датасет
# transforms.Compose — конвейер предобработки

# Предобработка: переводим в тензор + нормализуем
# Нормализация: pixel = (pixel - mean) / std
# mean и std — посчитаны по ВСЕМУ датасету CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),    # аугментация: отражение по горизонтали
    transforms.RandomCrop(32, padding=4), # аугментация: случайный кроп
    transforms.ToTensor(),                # [0, 255] → [0.0, 1.0]
    transforms.Normalize(                 # нормализация ImageNet-статистиками
        mean=[0.4914, 0.4822, 0.4465],   # среднее по CIFAR-10
        std=[0.2470, 0.2435, 0.2616]     # std по CIFAR-10
    ),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),                # без аугментаций!
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], 
                        std=[0.2470, 0.2435, 0.2616])
])

# Загружаем train и test
trainset = torchvision.datasets.CIFAR10(
    root='./data',       # куда скачивать
    train=True,          # тренировочный набор
    download=True,       # скачать, если нет
    transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,         # тестовый набор
    download=True,
    transform=transform_test
)

# DataLoader — «кормит» модель данными батчами
# batch_size=64 — 64 картинки за один шаг
# shuffle=True — перемешивать (важно для обучения!)
# num_workers=2 — 2 процесса загрузки данных (параллельно)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2
)

# Имена классов
classes = trainset.classes
print(f'Классы ({len(classes)}): {classes}')
print(f'Тренировочных картинок: {len(trainset)}')
print(f'Тестовых картинок: {len(testset)}')
print(f'Размер батча: 64')
print(f'Батчей (train): {len(trainloader)}')
print(f'Батчей (test): {len(testloader)}')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Примеры картинок CIFAR-10
# ============================================================
# Показываем по одному примеру каждого класса

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Находим по одной картинке каждого класса
shown = {}
for img, label in trainset:
    if label not in shown:
        shown[label] = img
    if len(shown) == 10:
        break

for i in range(10):
    ax = axes[i // 5][i % 5]
    # Денормализуем для отображения
    img = shown[i].numpy().transpose(1, 2, 0)
    img = img * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(classes[i], fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('CIFAR-10: 10 классов по одной картинке', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Картинки маленькие (32x32) — поэтому обучение будет быстрым!')
print('Но и модели сложнее различать детали на таком разрешении.')

---
## Шаг 8. Обучаем CNN

Теперь самое важное — обучить нашу модель! 

### Что нужно для обучения

| Компонент | Что делает | Наш выбор |
|-----------|-----------|----------|
| **Loss** | Измеряет ошибку | CrossEntropyLoss (мультиклассовая классификация) |
| **Optimizer** | Обновляет веса | Adam (адаптивный learning rate) |
| **Epoch** | Один проход по всем данным | 10 эпох |

### Цикл обучения

```
for each epoch:
    for each batch:
        1. Прямой проход: predictions = model(images)
        2. Вычислить loss: loss = criterion(predictions, labels)
        3. Обнулить градиенты: optimizer.zero_grad()
        4. Обратный проход: loss.backward()
        5. Обновить веса: optimizer.step()
```


In [ ]:
# ============================================================
# ШАГ 8: Обучаем CNN!
# ============================================================
# Создаём модель, loss и optimizer

model = SimpleCNN().to(device)  # пересоздаём для чистоты эксперимента

# CrossEntropyLoss = комбинация LogSoftmax + NLLLoss
# Вход: логиты (10 чисел), Цель: индекс класса (0-9)
criterion = nn.CrossEntropyLoss()

# Adam — адаптивный оптимизатор
# lr=0.001 — learning rate (размер «шага» при обновлении весов)
# weight_decay=1e-4 — L2-регуляризация (штраф за большие веса)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# ============================================================
# Цикл обучения
# ============================================================
num_epochs = 10

# Для графиков
train_losses = []
test_losses = []
train_accs = []
test_accs = []

print('Начинаем обучение...')
print(f'Эпох: {num_epochs}, Батчей: {len(trainloader)}, Устройство: {device}')
print()

for epoch in range(num_epochs):
    start_time = time.time()
    
    # ---- Тренировка ----
    model.train()  # режим обучения (Dropout работает)
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(trainloader):
        # Перемещаем данные на GPU/CPU
        images = images.to(device)
        labels = labels.to(device)
        
        # ШАГ 1: Прямой проход
        outputs = model(images)           # (64, 10) — логиты
        
        # ШАГ 2: Вычислить loss
        loss = criterion(outputs, labels)
        
        # ШАГ 3: Обнулить градиенты
        optimizer.zero_grad()
        
        # ШАГ 4: Обратный проход (вычислить градиенты)
        loss.backward()
        
        # ШАГ 5: Обновить веса
        optimizer.step()
        
        # Статистика
        running_loss += loss.item()
        _, predicted = outputs.max(1)      # индекс максимального логита
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = running_loss / len(trainloader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # ---- Тестирование ----
    model.eval()  # режим оценки (Dropout выключен)
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # НЕ вычислять градиенты (экономия памяти!)
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    test_loss = running_loss / len(testloader)
    test_acc = correct / total
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    elapsed = time.time() - start_time
    
    print(f'Эпоха {epoch+1:2d}/{num_epochs}: '
          f'train_loss={train_loss:.4f} train_acc={train_acc:.2%} | '
          f'test_loss={test_loss:.4f} test_acc={test_acc:.2%} | '
          f'{elapsed:.1f}s')

print('\n✅ Обучение завершено!')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Кривые обучения
# ============================================================
# Два графика: Loss и Accuracy по эпохам

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(train_losses, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(test_losses, 'r-o', label='Test Loss', markersize=4)
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss по эпохам', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(train_accs, 'b-o', label='Train Accuracy', markersize=4)
axes[1].plot(test_accs, 'r-o', label='Test Accuracy', markersize=4)
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Точность по эпохам', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Результат: test_acc = {test_accs[-1]:.2%}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Анализ
gap = train_accs[-1] - test_accs[-1]
print(f'Точность на тесте: {test_accs[-1]:.2%}')
print(f'Разрыв train-test: {gap:.2%}')
if gap > 0.10:
    print('⚠️ Большой разрыв — возможно переобучение! Попробуйте больше Dropout или data augmentation.')
elif gap > 0.05:
    print('⚠️ Небольшой разрыв — лёгкое переобучение. Нормально для простой модели.')
else:
    print('✅ Разрыв минимальный — модель хорошо обобщает!')

---
## Шаг 9. Оценка и визуализация предсказаний

Число точности — это хорошо, но давайте увидим **где именно** модель ошибается!


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Confusion Matrix
# ============================================================

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Строим confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(classes, fontsize=9)
ax.set_xlabel('Предсказание', fontsize=12)
ax.set_ylabel('Реальность', fontsize=12)
ax.set_title('Confusion Matrix — CIFAR-10', fontsize=14, fontweight='bold')

# Числа в ячейках
for i in range(10):
    for j in range(10):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# Какие классы путаются чаще всего?
print('Самые путаемые пары (реальность → предсказание):')
errors = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 30:
            errors.append((cm[i, j], classes[i], classes[j]))
errors.sort(reverse=True)
for count, real, pred in errors[:5]:
    print(f'  {real} → {pred}: {count} раз')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Правильные и ошибочные предсказания
# ============================================================

# Находим правильные и ошибочные предсказания
correct_idx = np.where(all_preds == all_labels)[0]
wrong_idx = np.where(all_preds != all_labels)[0]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# Верхний ряд: правильные предсказания
for i, ax in enumerate(axes[0]):
    idx = correct_idx[i * 100]  # берём разные
    img = testset.data[idx]
    ax.imshow(img)
    ax.set_title(f'✅ {classes[all_labels[idx]]}→{classes[all_preds[idx]]}', 
                fontsize=9, color='green')
    ax.axis('off')

# Нижний ряд: ошибки
for i, ax in enumerate(axes[1]):
    idx = wrong_idx[i]
    img = testset.data[idx]
    ax.imshow(img)
    ax.set_title(f'❌ {classes[all_labels[idx]]}→{classes[all_preds[idx]]}', 
                fontsize=9, color='red')
    ax.axis('off')

plt.suptitle('Верх: правильные ✅ | Низ: ошибки ❌', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Правильных: {len(correct_idx)} / {len(all_labels)} ({len(correct_idx)/len(all_labels):.1%})')
print(f'Ошибок: {len(wrong_idx)} / {len(all_labels)} ({len(wrong_idx)/len(all_labels):.1%})')
print('\nПосмотрите на ошибки — многие действительно неоднозначные!')

---
## Шаг 10. Что видит CNN? Визуализация фильтров и карт признаков

Самый интересный вопрос: **чему научилась сеть?** Какие паттерны она обнаруживает?

Мы визуализируем:
1. **Веса фильтров** первого свёрточного слоя — на что сеть «смотрит»
2. **Карты признаков** — что активируется для конкретной картинки


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ 1: Веса фильтров первого слоя
# ============================================================
# Первый слой видит «простые» паттерны: грани, цвета, текстуры
# Это аналоги наших ручных фильтров — но сеть УЧИЛАСЬ им сама!

# Извлекаем веса первого свёрточного слоя
# conv1.weight.shape = (16, 3, 3, 3)
# 16 фильтров, каждый 3x3, 3 канала (RGB)
filters = model.conv1.weight.data.cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Фильтры первого слоя Conv1 (чему научилась сеть?)', 
             fontsize=14, fontweight='bold')

for i in range(16):
    ax = axes[i // 8][i % 8]
    # Каждый фильтр — 3 канала RGB 3x3
    f = filters[i].numpy().transpose(1, 2, 0)  # (3,3,3) → (3,3,3)
    # Нормализуем для отображения
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f)
    ax.axis('off')
    ax.set_title(f'Фильтр {i}', fontsize=8)

plt.tight_layout()
plt.show()

print('Каждый квадратик — это фильтр 3x3x3 (RGB).')
print('Яркие области = веса, на которые фильтр «обращает внимание».')
print('Некоторые фильтры похожи на детекторы граней — как наши ручные!')
print('Другие — непонятные человеку, но полезные для сети.')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ 2: Карты признаков (Feature Maps)
# ============================================================
# Пропускаем картинку через сеть и смотрим, 
# какие карты признаков активируются на разных слоях

# Берём тестовую картинку (кошку)
cat_idx = None
for i in range(len(testset)):
    if testset.targets[i] == 3:  # 3 = cat в CIFAR-10
        cat_idx = i
        break

img_tensor = testset[cat_idx][0].unsqueeze(0).to(device)  # добавляем batch dim

# Показываем оригинал
img_np = testset[cat_idx][0].numpy().transpose(1, 2, 0)
img_np = img_np * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
img_np = np.clip(img_np, 0, 1)

fig, axes = plt.subplots(3, 9, figsize=(18, 6))

# Оригинал
axes[0][0].imshow(img_np)
axes[0][0].set_title('Оригинал', fontsize=10, fontweight='bold')
axes[0][0].axis('off')

# Карты признаков после conv1
model.eval()
with torch.no_grad():
    # Проходим через conv1 + relu
    x1 = F.relu(model.conv1(img_tensor))  # (1, 16, 32, 32)
    
    # Показываем 8 карт из 16
    for i in range(8):
        ax = axes[0][i+1] if i+1 < 9 else axes[1][i-7]
        if i+1 < 9:
            fm = x1[0, i].cpu().numpy()
            axes[0][i+1].imshow(fm, cmap='viridis')
            axes[0][i+1].set_title(f'Conv1\nкарта {i}', fontsize=8)
            axes[0][i+1].axis('off')

# Карты признаков после conv2
with torch.no_grad():
    x2 = F.relu(model.conv2(model.pool(x1)))  # (1, 32, 16, 16)
    
    for i in range(min(8, 32)):
        fm = x2[0, i].cpu().numpy()
        axes[1][i+1].imshow(fm, cmap='viridis')
        axes[1][i+1].set_title(f'Conv2\nкарта {i}', fontsize=8)
        axes[1][i+1].axis('off')

# Пустые ячейки
for i in range(9):
    axes[2][i].axis('off')
    
# Легенда
axes[2][0].text(0.5, 0.5, 'Conv1: простые\nпаттерны\n(грани, цвета)', 
               ha='center', va='center', fontsize=10, transform=axes[2][0].transAxes,
               bbox=dict(fc='lightyellow', ec='orange', boxstyle='round'))
axes[2][4].text(0.5, 0.5, 'Conv2: сложные\nпаттерны\n(формы, текстуры)', 
               ha='center', va='center', fontsize=10, transform=axes[2][4].transAxes,
               bbox=dict(fc='lightyellow', ec='orange', boxstyle='round'))

plt.suptitle('Карты признаков: что «видит» каждый слой', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('🔑 Ключевое наблюдение:')
print('  Conv1 (первый слой): видит ПРОСТЫЕ паттерны — грани, цвета, линии')
print('  Conv2 (второй слой): видит СЛОЖНЫЕ паттерны — формы, текстуры, комбинации')
print()
print('  Чем глубже слой — тем сложнее паттерны!')
print('  Это аналогично зрительной коре человека:')  
print('  V1 (простые клетки) → V2 (сложные клетки) → V4 (формы) → IT (объекты)')

---
## Шаг 11. Pooling, Stride, Padding — подробно с интерактивом

Эти три концепции управляют **размером** данных на выходе каждого слоя. Разберём каждый.


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Как работают Pooling, Stride, Padding
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ---- MAX POOLING ----
ax = axes[0]
ax.set_title('Max Pooling 2x2', fontsize=14, fontweight='bold', color='blue')
ax.axis('off')

# Показываем вход 4x4 и выход 2x2
input_pool = np.array([[1, 3, 2, 4],
                        [5, 6, 7, 8],
                        [9, 2, 1, 3],
                        [4, 5, 6, 0]])

for i in range(4):
    for j in range(4):
        ax.text(j+0.5, 3.5-i, str(input_pool[i,j]), ha='center', va='center', fontsize=12)

# Рамки 2x2 и результаты
colors = ['#FFB3BA', '#BAFFC9', '#BAE1FF', '#FFFFBA']
for idx, (r, c) in enumerate([(0,0), (0,2), (2,0), (2,2)]):
    ax.add_patch(plt.Rectangle((c, 3-r-1), 2, 2, fill=True, fc=colors[idx], 
                               ec='red', lw=2, alpha=0.3))
    val = input_pool[r:r+2, c:c+2].max()
    ax.text(c+1, 3-r-1-0.5, f'max={val}', ha='center', va='center', fontsize=9,
           color='red', fontweight='bold',
           bbox=dict(fc='white', ec='red', alpha=0.8, boxstyle='round'))

ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-1.5, 4.5)
ax.text(2, -1.2, 'Берёт максимум в окне 2x2\nУменьшает размер ВДВОЕ', 
       ha='center', fontsize=10, color='blue')

# ---- STRIDE ----
ax = axes[1]
ax.set_title('Stride = 1 vs 2', fontsize=14, fontweight='bold', color='green')
ax.axis('off')

# Stride=1
ax.text(2.5, 7.5, 'Stride=1 (каждый пиксель)', ha='center', fontsize=11, fontweight='bold')
for i in range(5):
    for j in range(5):
        ax.add_patch(plt.Rectangle((j, 6-i), 1, 1, fc='lightblue', ec='gray', lw=0.5))

for step in range(3):
    ax.add_patch(plt.Rectangle((step, 6), 3, 3, fill=False, ec='red', lw=2))
    ax.annotate('', xy=(step+1, 6), xytext=(step, 6),
               arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Stride=2
ax.text(2.5, 3.5, 'Stride=2 (через пиксель)', ha='center', fontsize=11, fontweight='bold')
for i in range(5):
    for j in range(5):
        ax.add_patch(plt.Rectangle((j, 2-i), 1, 1, fc='lightyellow', ec='gray', lw=0.5))

for step in range(2):
    ax.add_patch(plt.Rectangle((step*2, 2), 3, 3, fill=False, ec='green', lw=2))
    ax.annotate('', xy=(step*2+2, 2), xytext=(step*2, 2),
               arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-1.5, 8.5)
ax.text(2.5, -1.2, 'Stride=2: фильтр прыгает через 1 пиксель\nВыход в 2 раза меньше', 
       ha='center', fontsize=10, color='green')

# ---- PADDING ----
ax = axes[2]
ax.set_title('Padding = 0 vs 1', fontsize=14, fontweight='bold', color='purple')
ax.axis('off')

# Без padding
ax.text(2.5, 7.5, 'Padding=0 (без дополнения)', ha='center', fontsize=11, fontweight='bold')
for i in range(5):
    for j in range(5):
        ax.add_patch(plt.Rectangle((j, 6-i), 1, 1, fc='lightblue', ec='gray', lw=0.5))
ax.add_patch(plt.Rectangle((1, 4), 3, 3, fill=False, ec='red', lw=2))
ax.text(5.5, 5, 'Выход 3x3', fontsize=10, color='red')

# С padding
ax.text(3, 3.5, 'Padding=1 (нули по краям)', ha='center', fontsize=11, fontweight='bold')
for i in range(7):
    for j in range(7):
        color = '#FFB3BA' if i == 0 or i == 6 or j == 0 or j == 6 else 'lightblue'
        ax.add_patch(plt.Rectangle((j, 2-i), 1, 1, fc=color, ec='gray', lw=0.5))
for i in range(7):
    for j in range(7):
        if i == 0 or i == 6 or j == 0 or j == 6:
            ax.text(j+0.5, 2-i+0.5, '0', ha='center', va='center', fontsize=7, color='red')

ax.add_patch(plt.Rectangle((2, 0), 3, 3, fill=False, ec='purple', lw=2))
ax.text(7.5, 1, 'Выход 5x5\n(= входу!)', fontsize=10, color='purple')

ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-1.5, 8.5)
ax.text(3.5, -1.2, 'Padding=1 + kernel=3: размер НЕ меняется\n("same padding")', 
       ha='center', fontsize=10, color='purple')

plt.tight_layout()
plt.show()

---
## Шаг 12. Transfer Learning — предобученная модель vs наша

Наша SimpleCNN даёт ~70% точности. Можно ли лучше? Да — **transfer learning**!

Берём модель, обученную на ImageNet (миллионы картинок), и дообучаем на CIFAR-10.

### Почему transfer learning работает?

```
ImageNet (1.2 млн картинок, 1000 классов)
  → Модель научилась видеть: грани, текстуры, формы, объекты
  → Эти навыки ПЕРЕНОСЯТСЯ на другие задачи!

CIFAR-10 (50 тыс картинок, 10 классов)
  → Не учим с нуля, а «подкручиваем» уже обученную модель
  → Результат: ~95% вместо ~70%!
```


In [ ]:
# ============================================================
# Transfer Learning: ResNet18 предобученная на ImageNet
# ============================================================
# torchvision.models содержит предобученные модели
# resnet18 — 18-слойная сеть, обученная на 1.2 млн картинок ImageNet

import torchvision.models as models

# Загружаем предобученную ResNet18
# pretrained=True — скачиваем веса, обученные на ImageNet
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# ResNet18 ожидает вход 224x224, а CIFAR-10 — 32x32
# Нужно изменить первый слой и последний

# Заменяем первый conv: 7x7 stride=2 → 3x3 stride=1 (для маленьких картинок)
resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Убираем MaxPool (не нужен для 32x32)
resnet.maxpool = nn.Identity()  # «прозрачный» слой — ничего не делает

# Заменяем последний полносвязный слой: 1000 классов → 10 классов
num_features = resnet.fc.in_features  # 512
resnet.fc = nn.Linear(num_features, 10)

# Перемещаем на устройство
resnet = resnet.to(device)

# Считаем параметры
resnet_params = sum(p.numel() for p in resnet.parameters())
print(f'ResNet18 параметров: {resnet_params:,}')
print(f'SimpleCNN параметров: {total_params:,}')
print(f'Разница: {resnet_params/total_params:.1f}x')

# Обучаем ТОЛЬКО последний слой (быстрый transfer learning)
# Замораживаем все слои кроме fc
for name, param in resnet.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False  # НЕ обучать эти параметры

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'\nОбучаемых параметров: {trainable:,} (только последний слой!)')
print('Остальные — заморожены (используем как есть)')

In [ ]:
# ============================================================
# Обучаем ResNet18 (transfer learning)
# ============================================================
# Обучаем только последний слой — это БЫСТРО!

optimizer_tl = optim.Adam(resnet.fc.parameters(), lr=0.001)
criterion_tl = nn.CrossEntropyLoss()

num_epochs_tl = 5  # меньше эпох — модель уже почти готова!

train_accs_tl = []
test_accs_tl = []

print('Transfer Learning: обучаем ResNet18 на CIFAR-10...')
print()

for epoch in range(num_epochs_tl):
    resnet.train()
    # Замороженные слои остаются в режиме eval
    for module in resnet.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
    
    correct = 0
    total = 0
    running_loss = 0.0
    
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_tl.zero_grad()
        outputs = resnet(images)
        loss = criterion_tl(outputs, labels)
        loss.backward()
        optimizer_tl.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = correct / total
    train_accs_tl.append(train_acc)
    
    # Тест
    resnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    test_acc = correct / total
    test_accs_tl.append(test_acc)
    
    print(f'Эпоха {epoch+1}/{num_epochs_tl}: train_acc={train_acc:.2%} | test_acc={test_acc:.2%}')

print(f'\n✅ Transfer Learning завершён! Точность: {test_accs_tl[-1]:.2%}')

In [ ]:
# ============================================================
# СРАВНЕНИЕ: SimpleCNN vs ResNet18 (Transfer Learning)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].bar(['SimpleCNN\n(с нуля)', 'ResNet18\n(transfer)'],
           [test_accs[-1], test_accs_tl[-1]],
           color=['#4ECDC4', '#FF6B6B'], alpha=0.8, edgecolor='steelblue', lw=2)
axes[0].set_ylabel('Точность на тесте')
axes[0].set_title('Сравнение точности', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.0)

# Добавляем числа на столбцы
for i, (name, acc) in enumerate(zip(['SimpleCNN', 'ResNet18'], 
                                     [test_accs[-1], test_accs_tl[-1]])):
    axes[0].text(i, acc + 0.02, f'{acc:.1%}', ha='center', fontsize=14, fontweight='bold')

# Кривые обучения
axes[1].plot(train_accs, 'b-', label='SimpleCNN train', linewidth=2)
axes[1].plot(test_accs, 'b--', label='SimpleCNN test', linewidth=2)
axes[1].plot(train_accs_tl, 'r-', label='ResNet18 train', linewidth=2)
axes[1].plot(test_accs_tl, 'r--', label='ResNet18 test', linewidth=2)
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Точность')
axes[1].set_title('Кривые обучения', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('SimpleCNN vs ResNet18 Transfer Learning', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'SimpleCNN: {test_accs[-1]:.1%} (10 эпох, ~75K параметров)')
print(f'ResNet18:  {test_accs_tl[-1]:.1%} (5 эпох, ~11M параметров, ~5K обучаемых)')
print()
print('🔑 Вывод: предобученная модель быстрее обучается и точнее!')
print('Но SimpleCNN полезна для понимания — вы знаете КАЖДЫЙ её слой.')

---
## Шаг 13. Практические задания

Теперь ваша очередь! Экспериментируйте — ломайте код, чините, наблюдайте.


In [ ]:
# ============================================================
# ИНТЕРАКТИВНЫЙ: Список заданий
# ============================================================

tasks = widgets.Accordion(children=[
    widgets.HTML('''
        <h4>🎯 Задание 1: Измените архитектуру</h4>
        <p>Добавьте в SimpleCNN:</p>
        <ul>
          <li>Третий свёрточный слой <code>Conv2d(32, 64, 3, padding=1)</code></li>
          <li>Ещё один Pooling после него</li>
          <li>Не забудьте изменить <code>fc1</code> под новый размер!</li>
        </ul>
        <p><b>Вопрос:</b> Улучшилась ли точность? Насколько?</p>
        <p><b>Подсказка:</b> после 3-го pool размер будет 4x4, значит fc1 = 64*4*4 = 1024</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 2: Эксперименты с гиперпараметрами</h4>
        <p>Попробуйте по очереди:</p>
        <ul>
          <li><b>Learning rate:</b> 0.01 (много), 0.001 (ок), 0.0001 (мало)</li>
          <li><b>Batch size:</b> 16 (мало), 64 (ок), 256 (много)</li>
          <li><b>Dropout:</b> 0.0 (нет), 0.25 (ок), 0.5 (агрессивно)</li>
          <li><b>Optimizer:</b> SGD vs Adam — какой лучше?</li>
        </ul>
        <p><b>Записывайте результаты:</b> каждый эксперимент = 1 строка в таблице</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 3: Визуализация карт признаков для разных классов</h4>
        <p>Выберите картинки разных классов (самолёт, лягушка, грузовик) и:</p>
        <ul>
          <li>Пропустите через модель</li>
          <li>Покажите карты признаков conv1 и conv2</li>
          <li>Сравните: какие фильтры активируются для разных объектов?</li>
        </ul>
        <p><b>Подсказка:</b> используйте код из шага 10, замените cat_idx</p>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 4: Реализуйте Average Pooling вместо Max Pooling</h4>
        <p>Замените <code>nn.MaxPool2d(2)</code> на <code>nn.AvgPool2d(2)</code>:</p>
        <ul>
          <li>Как изменится точность?</li>
          <li>Как изменятся карты признаков?</li>
          <li>В каких задачах AvgPool лучше MaxPool?</li>
        </ul>
        <p><b>Код:</b></p>
        <pre>
# В __init__:
self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        </pre>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 5: Разморозьте все слои ResNet18</h4>
        <p>В шаге 12 мы заморозили все слои кроме последнего. Попробуйте:</p>
        <ol>
          <li><b>Вариант A:</b> Разморозьте ВСЕ слои и обучите с lr=0.0001</li>
          <li><b>Вариант B:</b> Разморозьте только layer4 + fc с lr=0.0001</li>
          <li>Сравните точность с замороженной версией</li>
        </ol>
        <p><b>Код для варианта A:</b></p>
        <pre>
for param in resnet.parameters():
    param.requires_grad = True
optimizer = optim.Adam(resnet.parameters(), lr=0.0001)
        </pre>
        <p><b>Внимание:</b> нужен маленький lr, иначе «разрушите» предобученные веса!</p>
    '''),
])

task_titles = [
    '🎯 Задание 1: Третий свёрточный слой',
    '🎯 Задание 2: Гиперпараметры',
    '🎯 Задание 3: Карты признаков для разных классов',
    '🎯 Задание 4: Average Pooling',
    '🎯 Задание 5: Разморозка ResNet18',
]
for i, title in enumerate(task_titles):
    tasks.set_title(i, title)

display(tasks)

---
## Что вы узнали

| Концепция | Что поняли |
|-----------|-----------|
| **Свёртка** | Фильтр скользит по картинке → карта признаков |
| **Фильтры** | Разные фильтры = разные паттерны (грани, текстуры) |
| **Pooling** | Уменьшает размер, сохраняет важное |
| **Stride** | Шаг фильтра: 1 = каждый пиксель, 2 = через один |
| **Padding** | Дополнение нулями: сохраняет размер выхода |
| **CNN архитектура** | Conv→ReLU→Pool→Conv→ReLU→Pool→FC→FC |
| **Feature maps** | Визуализация: что видит каждый слой |
| **Transfer Learning** | Предобученная модель = лучше + быстрее |

### Идеи для дальнейшего изучения

| Направление | Что изучить | Ресурс |
|------------|------------|--------|
| **Продвинутые архитектуры** | ResNet, VGG, Inception, EfficientNet | PyTorch Hub |
| **Детекция объектов** | YOLO, Faster R-CNN | Ultralytics |
| **Сегментация** | U-Net, Mask R-CNN | torchvision |
| **Генеративные модели** | GAN, VAE, Diffusion | PyTorch tutorials |
| **Оптимизация** | Квантизация, pruning, distillation | ONNX Runtime |

> Следующий блокнот: **RNN и обработка последовательностей** — как нейросети работают с текстом и временем.
